# 15 · Stage 3 v5-C — overlap inference + auxiliary-head fusion

이 notebook은 **재학습하지 않는다**. v5-B `best.pt`를 고정하고 추론 방식을 검증한다.

검증 순서:

1. comma2k19 validation에서 stop / accel / decel / turn / cruise가 섞인
   **완전한 segment 25개**를 deterministic하게 선택한다.
2. `T=32`에서 stride `32 / 16 / 8`을 비교한다.
3. overlap이 있는 경우 uniform average와 center-weighted overlap-add를 비교한다.
4. overlap 방법은 tune segment에서 고르되, untouched segment holdout에서
   non-overlap보다 나빠지면 reject한다.
5. 선택된 overlap feature에 STOP / accel ordinal / steering direction /
   yaw-turn auxiliary head를 작은 grid로 fusion한다.
6. fusion weight는 tune에서만 선택하고 holdout에서 다시 검증한다.
7. 마지막으로 released 50 labels는 **선택에 사용하지 않고** target-domain
   sanity check만 한다.

최종 출력 `v5c_selection.json`은 다음 submission builder가 그대로 사용한다.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(
            ["git", "-C", str(REPO), "checkout", BRANCH],
            check=True,
        )
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "easydict==1.13",
    ],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.dacon_inference import infer_public_frame_mapping
from blackbox_detection.stage3.metrics import dacon_stage3_metrics
from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.stage3.schema import read_frame_table
from blackbox_detection.stage3.v5c_inference import (
    FusionConfig,
    extract_video_features_v5c,
    grid_search_fusion,
    score_proxy_table,
    score_public_labels,
)
from blackbox_detection.utils import seed_everything
from blackbox_detection.utils.checkpoint import load_checkpoint

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
MANIFEST_ROOT = DRIVE_ROOT / "manifests/stage3/v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs/stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)

CFG_PATH = REPO / "configs/stage3/vjepa21b_can_v5c.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
V5B_CFG_PATH = REPO / cfg["experiment"]["source_config"]
v5b_cfg = yaml.safe_load(V5B_CFG_PATH.read_text(encoding="utf-8"))

RUN_NAME = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

print("GPU          :", torch.cuda.get_device_name(0))
print("v5-C config  :", CFG_PATH)
print("v5-B config  :", V5B_CFG_PATH)
print("output       :", RUN_DIR)
print("metric       : PASS")


## 1. Load v5-B best checkpoint as a frozen inference model


In [ ]:
def is_usable(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= min_bytes
    except OSError:
        return False

def copy_to_local(source: Path, dest: Path, min_bytes: int = 1):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".tmp")
    tmp.unlink(missing_ok=True)
    with source.open("rb") as src, tmp.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    if tmp.stat().st_size < min_bytes:
        raise OSError(f"staged file too small: {tmp.stat().st_size}")
    os.replace(tmp, dest)

VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"
if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/facebookresearch/vjepa2.git",
         str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT],
    check=True,
)

VJEPA_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_DRIVE = PRETRAINED_ROOT / VJEPA_NAME
VJEPA_LOCAL = LOCAL_PRETRAINED_ROOT / VJEPA_NAME
if not is_usable(VJEPA_LOCAL, 1_000_000_000):
    if not is_usable(VJEPA_DRIVE, 1_000_000_000):
        raise FileNotFoundError(VJEPA_DRIVE)
    copy_to_local(VJEPA_DRIVE, VJEPA_LOCAL, 1_000_000_000)

SOURCE_RUN = cfg["experiment"]["source_run"]
SOURCE_CKPT_DRIVE = (
    OUTPUT_ROOT / SOURCE_RUN / cfg["experiment"]["source_checkpoint"]
)
SOURCE_CKPT_LOCAL = (
    LOCAL_PRETRAINED_ROOT
    / f"{SOURCE_RUN}__{cfg['experiment']['source_checkpoint']}"
)
if not is_usable(SOURCE_CKPT_LOCAL, 1_000_000):
    if not is_usable(SOURCE_CKPT_DRIVE, 1_000_000):
        raise FileNotFoundError(SOURCE_CKPT_DRIVE)
    copy_to_local(SOURCE_CKPT_DRIVE, SOURCE_CKPT_LOCAL, 1_000_000)

from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.v5_models import VJEPA21DenseCANV5

dc = v5b_cfg["data"]
mc = v5b_cfg["model"]
fusion_model_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_LOCAL,
    num_frames=int(dc["clip_len"]),
    out_layers=tuple(mc["out_layers"]),
    freeze=True,
)

model = VJEPA21DenseCANV5(
    backbone,
    freeze_backbone=True,
    feature_dim=int(mc["feature_dim"]),
    temporal_hidden=int(mc["temporal_hidden"]),
    temporal_layers=int(mc["temporal_layers"]),
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=float(spatial_cfg["gate_init"]),
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=bool(fusion_model_cfg["enabled"]),
    accel_fusion_hidden=int(fusion_model_cfg["hidden"]),
    accel_fusion_gate_init=float(fusion_model_cfg["gate_init"]),
    accel_fusion_detach_ordinal_inputs=bool(
        fusion_model_cfg["detach_ordinal_inputs"]
    ),
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

meta = load_checkpoint(
    SOURCE_CKPT_LOCAL,
    model=model,
    optimizer=None,
    scheduler=None,
    map_location="cpu",
    strict=True,
    restore_rng_state=False,
)

device = torch.device("cuda")
model.to(device).eval()
model.requires_grad_(False)

STOP_THRESH = [float(x) for x in mc["stop_thresholds_mps"]]
ACCEL_THRESH = [float(x) for x in mc["accel_ordinal_thresholds_mps2"]]
TURN_THRESH = [float(x) for x in mc["turn_yaw_thresholds_rps"]]

print("checkpoint :", SOURCE_CKPT_LOCAL)
print("epoch      :", meta.get("epoch"))
print("T          :", dc["clip_len"])
print("stop thr   :", STOP_THRESH)
print("accel thr  :", ACCEL_THRESH)
print("turn thr   :", TURN_THRESH)


## 2. Build a diverse complete-segment validation subset

앞 800 window를 다시 쓰지 않는다. metadata만 훑어서 stop / hard decel /
hard accel / turn / cruise가 많은 segment를 각 5개씩 골라 **완전한 segment**
단위로 평가한다. 같은 segment의 일부 window가 tune과 holdout으로 갈라지지
않는다.


In [ ]:
val_manifest = pd.read_csv(MANIFEST_ROOT / "comma_val_id.csv")

def segment_profile(row):
    meta = read_frame_table(COMMA_ROOT / row.metadata_relpath)

    speed = meta["speed_mps"].to_numpy(dtype=np.float64)
    accel = meta["accel_from_speed_mps2"].to_numpy(dtype=np.float64)
    yaw = meta["yaw_rate_rps"].to_numpy(dtype=np.float64)

    vs = meta["valid_speed"].to_numpy(dtype=bool) & np.isfinite(speed)
    va = (
        meta["valid_accel_from_speed"].to_numpy(dtype=bool)
        & np.isfinite(accel)
    )
    vy = meta["valid_yaw"].to_numpy(dtype=bool) & np.isfinite(yaw)

    stop_fraction = float(np.mean(speed[vs] <= 1.0)) if vs.any() else 0.0
    hard_accel_fraction = (
        float(np.mean(accel[va] > 0.30)) if va.any() else 0.0
    )
    hard_decel_fraction = (
        float(np.mean(accel[va] < -0.30)) if va.any() else 0.0
    )
    turn_fraction = (
        float(np.mean(np.abs(yaw[vy]) > 0.03)) if vy.any() else 0.0
    )
    cruise_fraction = 0.0
    if vs.any() and va.any() and vy.any() and len(speed) == len(accel) == len(yaw):
        common = vs & va & vy
        if common.any():
            cruise_fraction = float(
                np.mean(
                    (speed[common] > 3.0)
                    & (np.abs(accel[common]) < 0.10)
                    & (np.abs(yaw[common]) < 0.01)
                )
            )

    return {
        "route_id": str(row.route_id),
        "segment_id": str(row.segment_id),
        "segment_key": f"{row.route_id}/{row.segment_id}",
        "num_frames": int(row.num_frames),
        "video_relpath": str(row.video_relpath),
        "metadata_relpath": str(row.metadata_relpath),
        "stop_start": stop_fraction,
        "hard_decel": hard_decel_fraction,
        "hard_accel": hard_accel_fraction,
        "turn": turn_fraction,
        "cruise": cruise_fraction,
    }

profiles = pd.DataFrame(
    segment_profile(row)
    for row in val_manifest.itertuples(index=False)
)

vc = cfg["validation"]
buckets = ["stop_start", "hard_decel", "hard_accel", "turn", "cruise"]
per_bucket = int(vc["per_bucket"])
num_segments = int(vc["num_segments"])

selected_rows = []
used = set()
for bucket in buckets:
    ranked = profiles.sort_values(
        [bucket, "segment_key"],
        ascending=[False, True],
    )
    taken = 0
    for _, row in ranked.iterrows():
        key = row["segment_key"]
        if key in used:
            continue
        item = row.to_dict()
        item["selection_bucket"] = bucket
        selected_rows.append(item)
        used.add(key)
        taken += 1
        if taken >= per_bucket:
            break

# Fill if bucket overlap prevented the requested total.
if len(selected_rows) < num_segments:
    remaining = profiles[~profiles["segment_key"].isin(used)].sort_values(
        "segment_key"
    )
    fill_n = num_segments - len(selected_rows)
    if fill_n > 0 and len(remaining):
        positions = np.linspace(
            0, len(remaining) - 1,
            num=min(fill_n, len(remaining)),
            dtype=int,
        )
        for pos in np.unique(positions):
            item = remaining.iloc[int(pos)].to_dict()
            item["selection_bucket"] = "coverage"
            selected_rows.append(item)

selected = pd.DataFrame(selected_rows).head(num_segments).copy()

# Stratified-by-selection-bucket holdout: roughly one of every 5 segments.
selected["split"] = "tune"
for bucket, idx in selected.groupby("selection_bucket", sort=False).groups.items():
    ids = list(idx)
    # Each main bucket contributes 5; reserve the last one.
    selected.loc[ids[-1], "split"] = "holdout"

# Guarantee at least 4 holdout segments.
if int((selected["split"] == "holdout").sum()) < 4:
    selected.loc[selected.index[::5], "split"] = "holdout"

selected_path = RUN_DIR / "selected_segments.csv"
selected.to_csv(selected_path, index=False)

print("selected:", len(selected))
print(selected["selection_bucket"].value_counts().to_dict())
print(selected["split"].value_counts().to_dict())
display(
    selected[
        [
            "segment_key", "selection_bucket", "split",
            "stop_start", "hard_decel", "hard_accel", "turn", "cruise"
        ]
    ]
)


## 3. Run overlap variants and cache dense features

각 stride의 model forward는 한 번만 수행하고, 같은 window prediction으로
`center_floor=1.0`과 `0.25`를 동시에 aggregate한다. 따라서 두 weighting
방식 비교 때문에 GPU 추론을 두 번 하지 않는다.

Drive cache가 있으면 런타임이 끊겨도 다시 추론하지 않는다.


In [ ]:
FEATURE_DIR = RUN_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

def attach_truth(pred, row):
    meta = read_frame_table(COMMA_ROOT / row.metadata_relpath).reset_index(drop=True)
    if len(meta) != len(pred):
        raise RuntimeError(
            f"frame count mismatch {row.segment_key}: "
            f"video={len(pred)}, metadata={len(meta)}"
        )
    out = pred.copy()
    out.insert(0, "segment_key", str(row.segment_key))
    out.insert(1, "route_id", str(row.route_id))
    out.insert(2, "segment_id", str(row.segment_id))
    out["gt_speed_mps"] = meta["speed_mps"].to_numpy(dtype=np.float32)
    out["gt_accel_mps2"] = meta[
        "accel_from_speed_mps2"
    ].to_numpy(dtype=np.float32)
    out["gt_steering_deg"] = meta["steering_deg"].to_numpy(dtype=np.float32)
    out["valid_speed"] = meta["valid_speed"].to_numpy(dtype=bool)
    out["valid_accel"] = meta[
        "valid_accel_from_speed"
    ].to_numpy(dtype=bool)
    out["valid_steer"] = meta["valid_steer"].to_numpy(dtype=bool)
    return out

strides = [int(x) for x in vc["strides"]]
floors = [float(x) for x in vc["center_floors"]]
method_tables = {}
runtime_rows = []

for stride in strides:
    cache_paths = {
        floor: FEATURE_DIR / f"features_s{stride}_c{floor:.2f}.csv"
        for floor in floors
    }
    if all(path.is_file() for path in cache_paths.values()):
        print(f"stride={stride}: CACHE HIT")
        for floor, path in cache_paths.items():
            method_tables[(stride, floor)] = pd.read_csv(path)
        continue

    per_floor_parts = {floor: [] for floor in floors}
    t0 = time.perf_counter()
    window_count = 0

    for i, row in enumerate(selected.itertuples(index=False), start=1):
        video_path = COMMA_ROOT / row.video_relpath
        variants = extract_video_features_v5c(
            model,
            video_path,
            target_stats=stats,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
            input_height=int(cfg["data"]["input_height"]),
            input_width=int(cfg["data"]["input_width"]),
            raw_stride=1,
            raw_offset=0,
            clip_len=int(cfg["data"]["clip_len"]),
            window_stride=stride,
            batch_size=int(cfg["data"]["batch_size"]),
            center_floors=floors,
            device=device,
            use_amp=bool(vc["use_amp"]),
        )
        n = len(next(iter(variants.values())))
        if n <= int(cfg["data"]["clip_len"]):
            windows = 1
        else:
            windows = (
                math.ceil(
                    (n - int(cfg["data"]["clip_len"])) / stride
                )
                + 1
            )
        window_count += windows

        for floor, table in variants.items():
            per_floor_parts[floor].append(attach_truth(table, row))

        if i == 1 or i % 5 == 0 or i == len(selected):
            print(f"stride={stride}: {i}/{len(selected)} segments")

    elapsed = time.perf_counter() - t0
    total_frames = int(selected["num_frames"].sum())
    runtime_rows.append(
        {
            "stride": stride,
            "seconds": elapsed,
            "frames": total_frames,
            "windows_approx": window_count,
            "seconds_per_600_frames": elapsed / total_frames * 600.0,
        }
    )

    for floor in floors:
        table = pd.concat(per_floor_parts[floor], ignore_index=True)
        table.to_csv(cache_paths[floor], index=False)
        method_tables[(stride, floor)] = table
        print("saved:", cache_paths[floor])

runtime_df = pd.DataFrame(runtime_rows)
display(runtime_df)


## 4. Compare overlap-only methods on tune / holdout / full


In [ ]:
proxy_rules = vc["proxy_rules"]
zero_fusion = FusionConfig()

def subset_for(table, split):
    keys = set(
        selected.loc[selected["split"] == split, "segment_key"].astype(str)
    )
    return table[table["segment_key"].astype(str).isin(keys)].reset_index(drop=True)

method_rows = []
method_scores = {}

for (stride, floor), table in method_tables.items():
    row = {
        "method": f"s{stride}_c{floor:.2f}",
        "stride": stride,
        "center_floor": floor,
    }
    for split_name in ("tune", "holdout"):
        part = subset_for(table, split_name)
        score = score_proxy_table(
            part,
            proxy_rules,
            fusion=zero_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )
        method_scores[(stride, floor, split_name)] = score
        row[f"{split_name}_stage3"] = score[
            "proxy/robust_mean_stage3_score"
        ]
        row[f"{split_name}_accel"] = score[
            "proxy/robust_mean_accel_macro_f1"
        ]
        row[f"{split_name}_steer"] = score[
            "proxy/robust_mean_steer_macro_f1"
        ]

    full = score_proxy_table(
        table,
        proxy_rules,
        fusion=zero_fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
    )
    method_scores[(stride, floor, "full")] = full
    row["full_stage3"] = full["proxy/robust_mean_stage3_score"]
    row["accel_corr"] = full["diag/accel/correlation"]
    row["accel_std_ratio"] = full[
        "diag/accel/pred_to_gt_std_ratio"
    ]
    row["accel_mae"] = full["diag/accel/mae"]
    method_rows.append(row)

method_df = pd.DataFrame(method_rows).sort_values(
    ["tune_stage3", "stride"],
    ascending=[False, False],
).reset_index(drop=True)

display(method_df)

baseline_key = (32, 1.0)
if baseline_key not in method_tables:
    raise RuntimeError("required baseline s32_c1.00 is missing")

best_tune = float(method_df["tune_stage3"].max())
tol = float(vc["overlap_tie_tolerance"])
eligible = method_df[
    method_df["tune_stage3"] >= best_tune - tol
].copy()

# Within statistical near-ties, prefer cheaper inference.
eligible = eligible.sort_values(
    ["stride", "tune_stage3"],
    ascending=[False, False],
)
chosen_overlap = eligible.iloc[0]

chosen_key = (
    int(chosen_overlap["stride"]),
    float(chosen_overlap["center_floor"]),
)

base_hold = method_scores[(*baseline_key, "holdout")][
    "proxy/robust_mean_stage3_score"
]
chosen_hold = method_scores[(*chosen_key, "holdout")][
    "proxy/robust_mean_stage3_score"
]

if chosen_hold < base_hold - float(vc["overlap_holdout_max_drop"]):
    print(
        "OVERLAP REJECTED by holdout:",
        chosen_key,
        "holdout", chosen_hold,
        "baseline", base_hold,
    )
    chosen_key = baseline_key
else:
    print("OVERLAP ACCEPTED:", chosen_key)

selected_features = method_tables[chosen_key]
print("selected overlap:", chosen_key)


## 5. Tune auxiliary fusion on tune segments only

192개의 작은 grid를 돈다. GPU inference는 다시 하지 않는다.

- STOP ordinal → speed boundary 주변 log-odds
- accel ordinal → ACCEL/DECEL boundary 주변 log-odds
- steering direction → continuous steering LEFT/RIGHT margin
- yaw-turn ordinal → 방향 보조 evidence

모든 weight가 0이면 continuous-only 결정과 **정확히 동일**하다.


In [ ]:
fc = cfg["fusion_search"]
tune_features = subset_for(selected_features, "tune")
hold_features = subset_for(selected_features, "holdout")

fusion_grid = grid_search_fusion(
    tune_features,
    proxy_rules,
    stop_thresholds_mps=STOP_THRESH,
    accel_thresholds_mps2=ACCEL_THRESH,
    turn_thresholds_rps=TURN_THRESH,
    stop_weights=fc["stop_weights"],
    accel_weights=fc["accel_weights"],
    steer_weights=fc["steer_weights"],
    turn_weights=fc["turn_weights"],
    stop_temperature_mps=fc["stop_temperature_mps"],
    accel_temperature_mps2=fc["accel_temperature_mps2"],
    steer_temperature_deg=fc["steer_temperature_deg"],
)

display(fusion_grid.head(20))

best_row = fusion_grid.iloc[0]
candidate = FusionConfig(
    stop_weight=float(best_row["stop_weight"]),
    accel_weight=float(best_row["accel_weight"]),
    steer_weight=float(best_row["steer_weight"]),
    turn_weight=float(best_row["turn_weight"]),
    stop_temperature_mps=float(best_row["stop_temperature_mps"]),
    accel_temperature_mps2=float(best_row["accel_temperature_mps2"]),
    steer_temperature_deg=float(best_row["steer_temperature_deg"]),
)

def scored(table, fusion):
    return score_proxy_table(
        table,
        proxy_rules,
        fusion=fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
    )

base_tune = scored(tune_features, zero_fusion)
base_hold = scored(hold_features, zero_fusion)
base_full = scored(selected_features, zero_fusion)

cand_tune = scored(tune_features, candidate)
cand_hold = scored(hold_features, candidate)
cand_full = scored(selected_features, candidate)

delta_tune = (
    cand_tune["proxy/robust_mean_stage3_score"]
    - base_tune["proxy/robust_mean_stage3_score"]
)
delta_hold = (
    cand_hold["proxy/robust_mean_stage3_score"]
    - base_hold["proxy/robust_mean_stage3_score"]
)
delta_full = (
    cand_full["proxy/robust_mean_stage3_score"]
    - base_full["proxy/robust_mean_stage3_score"]
)

hold_rule_deltas = {}
for name in proxy_rules:
    key = f"proxy/{name}/stage3_score"
    hold_rule_deltas[name] = cand_hold[key] - base_hold[key]

worst_hold_rule_delta = min(hold_rule_deltas.values())

accepted_fusion = (
    delta_hold >= float(fc["min_holdout_delta"])
    and delta_full >= float(fc["min_full_delta"])
    and worst_hold_rule_delta
        >= -float(fc["max_single_rule_holdout_drop"])
)

production_fusion = candidate if accepted_fusion else zero_fusion

comparison = pd.DataFrame(
    [
        {
            "variant": "overlap_only",
            "tune": base_tune["proxy/robust_mean_stage3_score"],
            "holdout": base_hold["proxy/robust_mean_stage3_score"],
            "full": base_full["proxy/robust_mean_stage3_score"],
            "accel": base_full["proxy/robust_mean_accel_macro_f1"],
            "steer": base_full["proxy/robust_mean_steer_macro_f1"],
        },
        {
            "variant": "candidate_aux_fusion",
            "tune": cand_tune["proxy/robust_mean_stage3_score"],
            "holdout": cand_hold["proxy/robust_mean_stage3_score"],
            "full": cand_full["proxy/robust_mean_stage3_score"],
            "accel": cand_full["proxy/robust_mean_accel_macro_f1"],
            "steer": cand_full["proxy/robust_mean_steer_macro_f1"],
        },
    ]
)
display(comparison)

print("candidate fusion:", candidate.as_dict())
print("holdout rule deltas:", hold_rule_deltas)
print("fusion accepted:", accepted_fusion)
print("production fusion:", production_fusion.as_dict())


## 6. Save v5-C selection


In [ ]:
selection_payload = {
    "version": 1,
    "source_run": SOURCE_RUN,
    "source_checkpoint": cfg["experiment"]["source_checkpoint"],
    "clip_len": int(cfg["data"]["clip_len"]),
    "overlap": {
        "stride": int(chosen_key[0]),
        "center_floor": float(chosen_key[1]),
    },
    "fusion": production_fusion.as_dict(),
    "fusion_candidate": candidate.as_dict(),
    "fusion_accepted": bool(accepted_fusion),
    "validation": {
        "num_segments": int(len(selected)),
        "tune_segments": int((selected["split"] == "tune").sum()),
        "holdout_segments": int((selected["split"] == "holdout").sum()),
        "overlap_only_tune": float(
            base_tune["proxy/robust_mean_stage3_score"]
        ),
        "overlap_only_holdout": float(
            base_hold["proxy/robust_mean_stage3_score"]
        ),
        "overlap_only_full": float(
            base_full["proxy/robust_mean_stage3_score"]
        ),
        "candidate_tune": float(
            cand_tune["proxy/robust_mean_stage3_score"]
        ),
        "candidate_holdout": float(
            cand_hold["proxy/robust_mean_stage3_score"]
        ),
        "candidate_full": float(
            cand_full["proxy/robust_mean_stage3_score"]
        ),
        "candidate_delta_tune": float(delta_tune),
        "candidate_delta_holdout": float(delta_hold),
        "candidate_delta_full": float(delta_full),
        "candidate_holdout_rule_deltas": {
            k: float(v) for k, v in hold_rule_deltas.items()
        },
    },
    "note": (
        "Proxy thresholds are diagnostic only. Fusion weights were selected "
        "on comma2k19 tune segments and gated on untouched comma2k19 holdout; "
        "released DACON labels are not used for selection."
    ),
}

selection_path = RUN_DIR / "v5c_selection.json"
selection_path.write_text(
    json.dumps(selection_payload, indent=2),
    encoding="utf-8",
)

method_df.to_csv(RUN_DIR / "overlap_comparison.csv", index=False)
fusion_grid.to_csv(RUN_DIR / "fusion_grid.csv", index=False)

print(json.dumps(selection_payload, indent=2))
print("saved:", selection_path)


## 7. Optional released-label sanity check

이 셀은 `Baseline.zip`의 공개 50 labels를 사용한다. **weight를 다시 고르지 않는다.**
이미 위에서 확정한 overlap/fusion을 그대로 적용해서 target-domain에서
catastrophic sign/threshold 문제가 없는지만 본다.


In [ ]:
if bool(cfg["public_sanity"]["enabled"]):
    baseline_candidates = [
        DRIVE_ROOT / "Baseline.zip",
        Path("/content/Baseline.zip"),
    ]
    baseline_zip = next(
        (p for p in baseline_candidates if p.is_file()),
        None,
    )

    if baseline_zip is None:
        print("PUBLIC SANITY SKIP: Baseline.zip not found")
    else:
        public_root = Path("/content/v5c_public_stage3")
        if public_root.exists():
            shutil.rmtree(public_root)
        public_root.mkdir(parents=True)

        with zipfile.ZipFile(baseline_zip) as zf:
            members = [
                n for n in zf.namelist()
                if "/data/stage3/" in n
            ]
            for name in members:
                zf.extract(name, public_root)

        labels_paths = list(public_root.rglob("data/stage3/labels.csv"))
        if len(labels_paths) != 1:
            raise RuntimeError(f"labels.csv candidates: {labels_paths}")
        labels_path = labels_paths[0]
        stage3_root = labels_path.parent
        labels = pd.read_csv(labels_path)

        video_paths = sorted((stage3_root / "videos").glob("*.mp4"))
        public_parts = []

        for video_path in video_paths:
            subset = labels[labels["ID"] == video_path.stem]
            raw_stride, raw_offset = infer_public_frame_mapping(subset)
            variants = extract_video_features_v5c(
                model,
                video_path,
                target_stats=stats,
                stop_thresholds_mps=STOP_THRESH,
                accel_thresholds_mps2=ACCEL_THRESH,
                turn_thresholds_rps=TURN_THRESH,
                input_height=int(cfg["data"]["input_height"]),
                input_width=int(cfg["data"]["input_width"]),
                raw_stride=raw_stride,
                raw_offset=raw_offset,
                clip_len=int(cfg["data"]["clip_len"]),
                window_stride=int(chosen_key[0]),
                batch_size=int(cfg["data"]["batch_size"]),
                center_floors=[float(chosen_key[1])],
                device=device,
                use_amp=False,
            )
            table = variants[float(chosen_key[1])].copy()
            table.insert(0, "ID", video_path.stem)
            public_parts.append(table)
            print(
                video_path.stem,
                "stride/offset", raw_stride, raw_offset,
                "10-Hz frames", len(table),
            )

        public_features = pd.concat(public_parts, ignore_index=True)
        cal_path = REPO / cfg["public_sanity"]["calibration"]
        calibration = json.loads(cal_path.read_text(encoding="utf-8"))

        public_no_aux = score_public_labels(
            public_features,
            labels,
            calibration,
            fusion=zero_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )
        public_selected = score_public_labels(
            public_features,
            labels,
            calibration,
            fusion=production_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )

        public_report = {
            "overlap_only": public_no_aux,
            "selected_v5c": public_selected,
            "delta_stage3": float(
                public_selected["stage3_score"]
                - public_no_aux["stage3_score"]
            ),
            "warning": (
                "Only 50 released labels. This is a sanity check, not a "
                "hyperparameter selection set."
            ),
        }
        (RUN_DIR / "public_sanity.json").write_text(
            json.dumps(public_report, indent=2),
            encoding="utf-8",
        )
        public_features.to_csv(
            RUN_DIR / "public_v5b_overlap_features.csv",
            index=False,
        )
        print(json.dumps(public_report, indent=2))


## 다음 단계

`v5c_selection.json`, `overlap_comparison.csv`, `public_sanity.json`(있으면)을
업로드한다. 그 결과를 보고 선택을 확정한 뒤 **Stage1 A7을 byte-lock한
v5-C submission builder**를 만든다.

공개 50 labels가 나쁘다고 여기서 grid를 다시 맞추지 않는다. 그 50개는 이미
기존 calibration에 사용된 작은 표본이므로 재튜닝하면 과적합 위험이 크다.
